In [2]:
%pip install pandas numpy matplotlib seaborn scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
print(pd.__version__)

3.1.0


In [4]:
df = pd.read_csv("E:\SQL_Projects\employee-attrition-analysis\data\HR-Employee-Attrition.csv")
print(df.shape)
df.head()

(1470, 35)


<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_31080\1640846075.py:1: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("E:\SQL_Projects\employee-attrition-analysis\data\HR-Employee-Attrition.csv")


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [6]:
df["Attrition"].value_counts(normalize=True) * 100

Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64

Baseline: 1,470 employees, 16.1% attrition (237 left, 1,233 stayed). The data is imbalanced (about 5:1), so accuracy alone will be misleading when we build the model.


In [7]:
print(df.isnull().sum().sum())
print(df.duplicated().sum())

0
0


In [8]:
for col in df.columns:
    if df[col].nunique() == 1:
        print(col, "->", df[col].unique())

EmployeeCount -> [1]
Over18 -> <StringArray>
['Y']
Length: 1, dtype: str
StandardHours -> [80]


In [9]:
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"])
print(df.shape)

(1470, 31)


In [10]:
df.to_csv("../data/hr_attrition_clean.csv", index=False)

In [11]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [12]:
df["AttritionFlag"]= (df["Attrition"] == "Yes").astype(int)

In [13]:
df["AttritionFlag"].mean()

np.float64(0.16122448979591836)

Tenure band

In [14]:
df["TenureBand"] = pd.cut(
    df["YearsAtCompany"],bins=[-1, 2, 5, 10, 50],
    labels=["0-2 yrs","3-5 yrs","6-10 yrs","10+ yrs"]
)

Age Band

In [15]:
df["AgeBand"] = pd.cut(
    df["Age"],
    bins=[17,25,35,45,60],
    labels=["18-25", "26-35","36-45","46-60"]
)

Income band

In [16]:
df["IncomeBand"] = pd.qcut(
    df["MonthlyIncome"],
    q=4,
    labels=["Low", "Lower-Mid", "Upper-Mid", "High"]
)

In [17]:
df["LongPromotionGap"] = (df["YearsSinceLastPromotion"] >= 4).astype(int)

In [18]:
df[["AttritionFlag", "TenureBand", "AgeBand", "IncomeBand", "LongPromotionGap"]].head()

,AttritionFlag,TenureBand,AgeBand,IncomeBand,LongPromotionGap
0,1,6-10 yrs,36-45,Upper-Mid,0
1,0,6-10 yrs,46-60,Upper-Mid,0
2,1,0-2 yrs,36-45,Low,0
3,0,6-10 yrs,26-35,Low,0
4,0,0-2 yrs,26-35,Lower-Mid,0


In [19]:
df.to_csv("../data/hr_attrition_clean.csv", index=False)

In [20]:
print(list(df.columns))

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'AttritionFlag', 'TenureBand', 'AgeBand', 'IncomeBand', 'LongPromotionGap']


In [21]:
%pip install pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
df.groupby("MaritalStatus")["Attrition"].agg(headcount="count")

,headcount
MaritalStatus,
Divorced,327
Married,673
Single,470


In [23]:
df.groupby("Attrition")["YearsAtCompany"].mean().round(1)

Attrition
No     7.4
Yes    5.1
Name: YearsAtCompany, dtype: float64

In [24]:
sub = df[(df["JobRole"] == "Sales Representative") & (df["OverTime"] == "Yes")]
print(len(sub), round(sub["AttritionFlag"].mean() * 100, 1))

24 66.7


In [25]:
df.groupby("JobRole")["MonthlyIncome"].agg(["count","mean","median","min","max"])

,count,mean,median,min,max
JobRole,,,,,
Healthcare Representative,131,7528.763359,6811.0,4000,13966
Human Resources,52,4235.750000,3093.0,1555,10725
Laboratory Technician,259,3237.169884,2886.0,1102,7403
Manager,102,17181.676471,17454.5,11244,19999
Manufacturing Director,145,7295.137931,6447.0,4011,13973
Research Director,80,16033.550000,16510.0,11031,19973
Research Scientist,292,3239.972603,2887.5,1009,9724
Sales Executive,326,6924.279141,6231.0,4001,13872
Sales Representative,83,2626.000000,2579.0,1052,6632


In [26]:
df.groupby("Department").agg(
    headcount=("EmployeeNumber" if "EmployeeNumber" in df else "Age","count"),
    avg_income=("MonthlyIncome","mean"),
    avg_tenure=("YearsAtCompany","mean"),
    attrition_pct=("AttritionFlag","mean")
)

,headcount,avg_income,avg_tenure,attrition_pct
Department,,,,
Human Resources,63,6654.507937,7.238095,0.190476
Research & Development,961,6281.252862,6.864724,0.138398
Sales,446,6959.172646,7.284753,0.206278


In [27]:
df.groupby(["Department","OverTime"])["AttritionFlag"].agg(["count","mean"])

count      mean
Department             OverTime                 
Human Resources        No           46  0.152174
                       Yes          17  0.294118
Research & Development No          690  0.085507
                       Yes         271  0.273063
Sales                  No          318  0.138365
                       Yes         128  0.375000

In [28]:
g=df.groupby("JobRole")["AttritionFlag"].agg(headcount="count",rate="mean")
g[g["headcount"]>100]

,headcount,rate
JobRole,,
Healthcare Representative,131,0.068702
Laboratory Technician,259,0.239382
Manager,102,0.049020
Manufacturing Director,145,0.068966
Research Scientist,292,0.160959
Sales Executive,326,0.174847


In [29]:
df.groupby("Gender").agg(
    avg_age=("Age","mean"),
    avg_income=("MonthlyIncome","mean")
)

,avg_age,avg_income
Gender,,
Female,37.329932,6686.566327
Male,36.653061,6380.507937


In [30]:
result = df.groupby("EducationField").size().to_frame("emp")
result["pct"] = (result["emp"]/result["emp"].sum()*100).round(1)
result = result.sort_values("pct",ascending=False)
result


,emp,pct
EducationField,,
Life Sciences,606,41.2
Medical,464,31.6
Marketing,159,10.8
Technical Degree,132,9.0
Other,82,5.6
Human Resources,27,1.8


In [31]:
(df.groupby("BusinessTravel")["AttritionFlag"]
   .agg(headcount="count", attrition_pct="mean")
   .assign(attrition_pct=lambda d: (d["attrition_pct"] * 100).round(1))
   .sort_values("attrition_pct", ascending=False))


,headcount,attrition_pct
BusinessTravel,,
Travel_Frequently,277,24.9
Travel_Rarely,1043,15.0
Non-Travel,150,8.0


In [32]:
df.groupby(["Department","Gender"])["AttritionFlag"].agg("mean")

Department              Gender
Human Resources         Female    0.300000
                        Male      0.139535
Research & Development  Female    0.113456
                        Male      0.154639
Sales                   Female    0.201058
                        Male      0.210117
Name: AttritionFlag, dtype: float64

In [33]:
g = df.groupby("JobRole")["AttritionFlag"].agg(headcount="count", attrition_pct="mean")
g["attrition_pct"] = (g["attrition_pct"] * 100).round(1)

g[g["headcount"] > 100].sort_values("attrition_pct", ascending=False)

,headcount,attrition_pct
JobRole,,
Laboratory Technician,259,23.9
Sales Executive,326,17.5
Research Scientist,292,16.1
Healthcare Representative,131,6.9
Manufacturing Director,145,6.9
Manager,102,4.9


In [34]:
(df.groupby(["Department", "Attrition"])["MonthlyIncome"]
   .agg(headcount="count", median_income="median")
   .round(0))

headcount  median_income
Department             Attrition                          
Human Resources        No                51         4936.0
                       Yes               12         2652.0
Research & Development No               828         4708.0
                       Yes              133         2759.0
Sales                  No               354         6074.0
                       Yes               92         5314.0

In [38]:
df["DepAvgIncome"]=df.groupby("Department")["MonthlyIncome"].transform("mean")
below = df[df["MonthlyIncome"]<df["DepAvgIncome"]]
print(len(below))
below["AttritionFlag"].mean() * 100

987


np.float64(19.148936170212767)

In [42]:
(df.sort_values("MonthlyIncome", ascending=False)
   .groupby("JobRole")
   .head(2)
   .sort_values(["JobRole", "MonthlyIncome"], ascending=[True, False])
   [["JobRole", "MonthlyIncome", "Attrition"]])

,JobRole,MonthlyIncome,Attrition
1181,Healthcare Representative,13966,No
316,Healthcare Representative,13964,No
510,Human Resources,10725,No
789,Human Resources,10482,Yes
677,Laboratory Technician,7403,No
816,Laboratory Technician,6782,No
190,Manager,19999,No
851,Manager,19943,No
721,Manufacturing Director,13973,No
627,Manufacturing Director,13826,No


In [43]:
overall=df["AttritionFlag"].mean()
g=df.groupby("Department")["AttritionFlag"].mean()
g[g>overall]

Department
Human Resources    0.190476
Sales              0.206278
Name: AttritionFlag, dtype: float64

In [52]:
df["RoleAvgIncome"]=df.groupby("JobRole")["MonthlyIncome"].transform("mean")
below=df[(df["Attrition"]=="Yes") &(df["MonthlyIncome"]<df["RoleAvgIncome"])]
below.groupby("JobRole").size().sort_values(ascending=False)

JobRole
Laboratory Technician        42
Research Scientist           36
Sales Executive              31
Sales Representative         19
Human Resources              10
Manufacturing Director        5
Manager                       2
Healthcare Representative     2
dtype: int64

In [54]:
both =(df["OverTime"]=="Yes")&(df["IncomeBand"]=="Low")
either = (df["OverTime"]=="Yes")&(df["IncomeBand"]=="Low")
df["RiskFlag"] = np.select([both,either],["High","Medium"], default="Low")
df.groupby("RiskFlag")["AttritionFlag"].agg(headcount="count",attrition_pct="mean")

,headcount,attrition_pct
RiskFlag,,
High,106,0.584906
Low,1364,0.128299


In [55]:
df.pivot_table(index="Department", columns="OverTime",
               values="AttritionFlag", aggfunc="mean")

OverTime,No,Yes
Department,,
Human Resources,0.152174,0.294118
Research & Development,0.085507,0.273063
Sales,0.138365,0.375000


In [56]:
df.pivot_table(index="JobRole", columns="OverTime",
               values="AttritionFlag", aggfunc=["count", "mean"])

count          mean          
OverTime                     No Yes        No       Yes
JobRole                                                
Healthcare Representative    94  37  0.074468  0.054054
Human Resources              39  13  0.179487  0.384615
Laboratory Technician       197  62  0.157360  0.500000
Manager                      75  27  0.013333  0.148148
Manufacturing Director      106  39  0.056604  0.102564
Research Director            57  23  0.017544  0.043478
Research Scientist          195  97  0.071795  0.340206
Sales Executive             232  94  0.112069  0.329787
Sales Representative         59  24  0.288136  0.666667